# Faster Whisper Inference with torch.compile

This notebook demonstrates how to speed up Whisper inference using PyTorch 2.0's `torch.compile` feature. The `torch.compile` API can provide significant speedups by optimizing model execution through graph compilation and kernel fusion.

We'll compare baseline inference performance with compiled versions and explore different compilation modes and strategies.

In [ ]:
# Install required packages
!pip install transformers torch torchaudio datasets

## Load Model and Processor

We'll use the `openai/whisper-small` model for this demonstration. This model provides a good balance between accuracy and speed.

In [ ]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import time

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load model and processor
model_id = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_id)
model = WhisperForConditionalGeneration.from_pretrained(model_id)
model = model.to(device)

print(f"Model loaded: {model_id}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

## Baseline Inference

First, let's establish a baseline by running inference without any optimizations. We'll use a sample from the LibriSpeech dataset.

In [ ]:
from datasets import load_dataset
import torchaudio

# Load sample audio
ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
sample = ds[0]

# Prepare audio input
audio_array = sample["audio"]["array"]
sampling_rate = sample["audio"]["sampling_rate"]

print(f"Audio sample rate: {sampling_rate} Hz")
print(f"Audio duration: {len(audio_array) / sampling_rate:.2f} seconds")
print(f"Reference text: {sample['text']}")

# Process audio
inputs = processor(audio_array, sampling_rate=sampling_rate, return_tensors="pt")
inputs = inputs.input_features.to(device)

# Baseline inference with timing
model.eval()
with torch.no_grad():
    start_time = time.time()
    generated_ids = model.generate(inputs)
    baseline_time = time.time() - start_time

transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(f"\nBaseline transcription: {transcription}")
print(f"Baseline inference time: {baseline_time:.4f} seconds")

## Apply torch.compile

Now let's apply `torch.compile` to optimize the model. PyTorch 2.0 offers different compilation modes:
- `default`: Balanced optimization
- `reduce-overhead`: Minimize overhead, good for small batches
- `max-autotune`: Maximum performance, longer compilation time

We'll compile the entire model for optimal performance.

In [ ]:
# Apply torch.compile to the model
compiled_model = torch.compile(model, mode="reduce-overhead")

print("Model compiled with torch.compile (mode='reduce-overhead')")
print("Note: First inference will be slower due to compilation warmup")

# Warmup run - this triggers compilation
print("\nPerforming warmup run...")
with torch.no_grad():
    warmup_start = time.time()
    _ = compiled_model.generate(inputs)
    warmup_time = time.time() - warmup_start
print(f"Warmup time (includes compilation): {warmup_time:.4f} seconds")

# Actual compiled inference
with torch.no_grad():
    start_time = time.time()
    compiled_generated_ids = compiled_model.generate(inputs)
    compiled_time = time.time() - start_time

compiled_transcription = processor.batch_decode(compiled_generated_ids, skip_special_tokens=True)[0]
print(f"\nCompiled transcription: {compiled_transcription}")
print(f"Compiled inference time: {compiled_time:.4f} seconds")
print(f"Speedup: {baseline_time / compiled_time:.2f}x")

## Benchmarking

Let's run multiple iterations to get more reliable performance measurements and compare baseline vs compiled models.

In [ ]:
import numpy as np

def benchmark_model(model, inputs, num_runs=10, warmup_runs=3):
    """Benchmark model inference with multiple runs"""
    times = []
    
    # Warmup
    for _ in range(warmup_runs):
        with torch.no_grad():
            _ = model.generate(inputs)
    
    # Benchmark
    for i in range(num_runs):
        if device == "cuda":
            torch.cuda.synchronize()
        
        start_time = time.time()
        with torch.no_grad():
            _ = model.generate(inputs)
        
        if device == "cuda":
            torch.cuda.synchronize()
        
        times.append(time.time() - start_time)
    
    return times

# Benchmark both models
print("Benchmarking baseline model...")
baseline_times = benchmark_model(model, inputs, num_runs=10)

print("Benchmarking compiled model...")
compiled_times = benchmark_model(compiled_model, inputs, num_runs=10)

# Calculate statistics
baseline_mean = np.mean(baseline_times)
baseline_std = np.std(baseline_times)
compiled_mean = np.mean(compiled_times)
compiled_std = np.std(compiled_times)

print("\n" + "="*60)
print("BENCHMARK RESULTS")
print("="*60)
print(f"Baseline model:")
print(f"  Mean: {baseline_mean:.4f} s")
print(f"  Std:  {baseline_std:.4f} s")
print(f"  Min:  {min(baseline_times):.4f} s")
print(f"  Max:  {max(baseline_times):.4f} s")
print(f"\nCompiled model (torch.compile):")
print(f"  Mean: {compiled_mean:.4f} s")
print(f"  Std:  {compiled_std:.4f} s")
print(f"  Min:  {min(compiled_times):.4f} s")
print(f"  Max:  {max(compiled_times):.4f} s")
print(f"\nSpeedup: {baseline_mean / compiled_mean:.2f}x")
print(f"Latency reduction: {(1 - compiled_mean / baseline_mean) * 100:.1f}%")
print("="*60)

## Using with Pipeline

The Hugging Face `pipeline` API provides a convenient high-level interface. We can pass our compiled model to the pipeline for easy use.

In [ ]:
from transformers import pipeline
import numpy as np

# Create pipeline with baseline model
baseline_pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=device
)

# Create pipeline with compiled model
compiled_pipe = pipeline(
    "automatic-speech-recognition",
    model=compiled_model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=device
)

# Prepare audio input as numpy array for pipeline
audio_input = np.array(sample["audio"]["array"])

# Test baseline pipeline
start_time = time.time()
baseline_result = baseline_pipe(audio_input)
baseline_pipe_time = time.time() - start_time

# Test compiled pipeline
start_time = time.time()
compiled_result = compiled_pipe(audio_input)
compiled_pipe_time = time.time() - start_time

print("Pipeline Results:")
print(f"\nBaseline pipeline:")
print(f"  Text: {baseline_result['text']}")
print(f"  Time: {baseline_pipe_time:.4f} s")
print(f"\nCompiled pipeline:")
print(f"  Text: {compiled_result['text']}")
print(f"  Time: {compiled_pipe_time:.4f} s")
print(f"\nPipeline speedup: {baseline_pipe_time / compiled_pipe_time:.2f}x")

## Tips and Best Practices

### Compilation Modes
- **`default`**: Good starting point, balanced performance and compilation time
- **`reduce-overhead`**: Best for inference workloads with small batch sizes
- **`max-autotune`**: Maximum performance but longer compilation time (minutes)

### Warmup
- Always perform warmup runs after compilation
- First inference triggers actual compilation and will be slow
- Subsequent runs benefit from cached compiled graphs

### Padding Strategies
- Use consistent input shapes for best performance
- Variable-length inputs may trigger recompilation
- Consider bucketing audio lengths in production

### Hardware Considerations
- Speedups are typically larger on GPU than CPU
- CUDA graphs provide additional optimization on NVIDIA GPUs
- PyTorch 2.0+ required for `torch.compile`

### Memory Usage
- Compiled models may use slightly more memory
- Graph caching persists across runs
- Use `torch._dynamo.reset()` to clear compiled cache if needed

### Production Deployment
- Pre-compile models before serving
- Save compilation time by doing warmup during initialization
- Consider batching requests for better throughput
- Monitor memory usage and adjust batch sizes accordingly

### Debugging
- Use `torch._dynamo.explain()` to understand compilation behavior
- Set `TORCH_LOGS="+dynamo"` environment variable for detailed logs
- Check for graph breaks that may limit optimization